# M6.A2 — EDA: 판매·외부 데이터 분포·결측·이상치 정량화

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A2 · `docs/research/ai/00_ml_guide_reference.md` PART 1
> 입력: `AI/data/processed/` + `AI/data/manual/` (M6.A1 적재분, 카탈로그: `AI/data/README.md`)
> 작성: 2026-07-27

**목적** — 타깃(일별 매출)과 피처 소스(기상·공휴일·학사일정·유동인구)의 분포·결측·이상치를 정량화하고,
타깃과의 관계를 확인해 M6.A3(피처 엔지니어링)·M6.A4(결측·이상치 규칙 확정)의 입력을 만든다.

**누수 방지 원칙** (`08_ai/ml_pipeline.md` §6) — 본 노트북의 모든 탐색은 *진단용*이다.
전 구간 데이터를 자유롭게 보되, 보간·임계값 등 **모델용 변환의 확정은 M6.A4에서 train 구간 기준**으로만 한다.

**실행 방법**
```bash
# 모델링 env 준비는 AI/README.md 참조 ([ml] extra)
cd AI/notebooks
jupyter nbconvert --to notebook --execute --inplace 01_eda.ipynb
```

> 🔒 **공개 저장소 데이터 정책** — 실매장 매출의 절대 금액(원 단위 수치·그림·표)은 커밋하지 않는다. 본 노트북은 **출력 제거 상태로 추적**되며, 본문 서술의 금액은 비율·배수로 대체했다. 전체 수치·그림은 로컬 재실행으로 전량 재현된다 (실행법: `AI/README.md`).

## 판정 요약 (TL;DR)

1. **타깃 확정치** — 영업일 **256일**(첫 매출 2025-04-10 ~ 2026-04-16), 일 매출은 우측 왜도 분포(skew 1.49). 야간 영업 주점(매출의 40%가 주류, 1위 메뉴 소주).
2. **무매출 123일 구조 분해** — 개업 전 7 + 여름 휴가 6 + 추석 연휴 9 + **장기 휴업 67**(2025-12-21~2026-02-25) + 산발 34일(일요일 위주).
   **정기휴무 요일 없음** (일요일 휴무율 42%, 나머지 요일 7~18%).
3. **장기 휴업 후 사실상 재런칭** — 재개장 후 판매 메뉴 90종 중 **69종 신규**('골목 낙곱새' 시리즈로 주력 교체), 유지 21종.
   일 매출 중앙값 **2.2배**. 마라 중심 → 낙곱새 중심 업종 개편으로 판단됨
   → **학습 구간 설계에 결정적. 담당자 검수(휴업 사유·운영 변경)와 직결** (§2.6, §9).
4. **주간 주기 뚜렷** — 목요일 피크·일요일 저점(중앙값 2.3배 차), 7일 자기상관 0.47 → lag7·rolling7 피처 유효.
5. **학기 효과 큼** — 휴업 전 구간만으로도 학기중 중앙값이 방학기의 **2.2배**, Spearman 0.36.
6. **기온 음의 상관(-0.35)은 단독 해석 금지** — 여름 방학 슬럼프와 얽혀 있어 M6.A3 다변량에서 재평가.
   강수는 억제 효과 없음. 시험주간은 학기중 대비 뚜렷이 낮음(휴업 전 기준 중앙값 34%↓ — 단 일자가 estimated, 검수 대기).
7. **데이터 품질** — 기상 결측은 판매 기간 내 NA≤2로 무시 수준. 메뉴 `menu_clean` NaN 7행·표기 이형 1그룹
   → 매핑 확정안 §3. 유동인구 누락 7개월·2025-08 생활인구 0 오류(검수 대기)는 월간 참고 피처라 비차단.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 데이터 경로 — notebooks/ 또는 저장소 루트 어디서 실행해도 동작
_here = Path.cwd()
DATA = next(p / "data" for p in [_here.parent, _here, _here / "AI"] if (p / "data" / "processed").exists())

# 시각화 규칙 — 고정 팔레트(파랑=주 시리즈, 주황=대비 시리즈), 격자·테두리 최소화
PAL = {"blue": "#2a78d6", "blue_dark": "#1c5cab", "orange": "#eb6834",
       "aqua": "#1baf7a", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
_kr_fonts = [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic", "Malgun Gothic")
             if f in _installed] or ["sans-serif"]
plt.rcParams.update({
    "font.family": _kr_fonts,
    "axes.unicode_minus": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "figure.dpi": 100,
})

tot  = pd.read_csv(DATA / "processed/sales_daily_total.csv", parse_dates=["date"])
menu = pd.read_csv(DATA / "processed/sales_daily_menu.csv", parse_dates=["date"])
wx   = pd.read_csv(DATA / "processed/weather_daily.csv", parse_dates=["date"])
hol  = pd.read_csv(DATA / "processed/holidays.csv", parse_dates=["date"])
acad = pd.read_csv(DATA / "manual/academic_calendar.csv", parse_dates=["start_date", "end_date"])
pop  = pd.read_csv(DATA / "processed/population_monthly.csv")

won = lambda x: f"{x:,.0f}"  # 표 표시용 원 단위 콤마

pd.DataFrame(
    [
        ("sales_daily_total", len(tot), f"{tot.date.min():%Y-%m-%d} ~ {tot.date.max():%Y-%m-%d}", int(tot.date.duplicated().sum())),
        ("sales_daily_menu", len(menu), f"{menu.date.min():%Y-%m-%d} ~ {menu.date.max():%Y-%m-%d}", int(menu.duplicated().sum())),
        ("weather_daily", len(wx), f"{wx.date.min():%Y-%m-%d} ~ {wx.date.max():%Y-%m-%d}", int(wx.duplicated(["station_id", "date"]).sum())),
        ("holidays", len(hol), f"{hol.date.min():%Y-%m-%d} ~ {hol.date.max():%Y-%m-%d}", int(hol.date.duplicated().sum())),
        ("academic_calendar", len(acad), f"{acad.start_date.min():%Y-%m-%d} ~ {acad.end_date.max():%Y-%m-%d}", int(acad.duplicated().sum())),
        ("population_monthly", len(pop), f"{pop.month.min()} ~ {pop.month.max()}", int(pop.duplicated(["month", "dong"]).sum())),
    ],
    columns=["파일", "행수", "기간", "중복키"],
)

### §1 관찰 — 로드·정합성

- 6개 소스 모두 중복 키 0. 기간·행수는 카탈로그(`AI/data/README.md`) 기재와 일치.
- 기상은 4개 관측소 병합본 → 분석은 조치원 최근접 **세종연서(611)** 만 사용.
- 유동인구는 세종 전체 행정동 포함 → **조치원읍**만 사용. 학사일정은 2020~2026 전체 → 판매 겹침 구간만 사용.

## §2 타깃 — 일별 매출

**영업일 정의** — `total_amount > 0`인 날. 리포트 수록 258행 중 0원이 2행 있어 수록일≠영업일이다.
파일에 없는 캘린더 날짜(121일)는 POS 리포트에 매출 행이 없는 날 = 무매출일로 간주한다.

특수일 2건:
- **2025-04-03** — 수록됐지만 0원. 메뉴 행이 `라면`·`테스트` (qty 0)뿐 → **POS 설치 테스트일**. 실질 개업일은 첫 매출일 **2025-04-10**.
- **2026-03-22(일)** — 재개장 후 유일한 0원 수록일. 재개장 후 첫 휴무(일요일)로 해석.

In [ ]:
CAL = pd.date_range(tot.date.min(), tot.date.max(), freq="D")
full = tot.set_index("date").reindex(CAL).rename_axis("date")
full["total_amount"] = full["total_amount"].fillna(0)
full["tx_count"] = full["tx_count"].fillna(0)
full["in_report"] = full["source_file"].notna()
full["is_open"] = full["total_amount"] > 0

OPEN_DAY = pd.Timestamp("2025-04-10")                                 # 첫 매출일 = 실질 개업일
CLOSE_S, CLOSE_E = pd.Timestamp("2025-12-21"), pd.Timestamp("2026-02-25")  # 장기 휴업
REOPEN = pd.Timestamp("2026-02-26")

# 연속 무매출 구간 (5일 이상)
closed = ~full.is_open
grp = (closed != closed.shift()).cumsum()
runs = [(g.index[0], g.index[-1], len(g)) for _, g in full[closed].groupby(grp[closed]) if len(g) >= 5]

print(f"캘린더 {len(CAL)}일 | 리포트 수록 {int(full.in_report.sum())}일 "
      f"| 영업일(매출>0) {int(full.is_open.sum())}일 | 무매출 {int(closed.sum())}일 "
      f"(파일 미수록 {int((~full.in_report).sum())} + 0원 수록 {int((full.in_report & ~full.is_open).sum())})")
print("\n리포트에 수록됐지만 매출 0원인 날:")
z = tot[tot.total_amount == 0].copy()
z["요일"] = z.date.dt.dayofweek.map(dict(enumerate("월화수목금토일")))
display(z[["date", "total_amount", "tx_count", "요일", "source_file"]])

In [ ]:
# 전 기간 시계열 — 일별 매출(만원) + 7일 이동평균, 무매출 연속 구간 음영
fig, ax = plt.subplots(figsize=(12.5, 4.4), constrained_layout=True)
for s, e, n in runs:
    ax.axvspan(s, e + pd.Timedelta(days=1), color=PAL["gray"], alpha=0.55, zorder=0)
ax.plot(full.index, full.total_amount / 1e4, lw=0.9, color=PAL["blue"], alpha=0.85, label="일별 매출")
ma7 = full.total_amount.rolling(7, min_periods=4).mean()
ax.plot(full.index, ma7 / 1e4, lw=2, color=PAL["blue_dark"], label="7일 이동평균")

notes = [(OPEN_DAY, "개업 04-10", 0.78), (pd.Timestamp("2025-08-06"), "여름 휴가", 0.97),
         (pd.Timestamp("2025-10-04"), "추석 연휴", 0.97), (pd.Timestamp("2026-01-05"), "장기 휴업 67일", 0.97),
         (REOPEN, "재개장 02-26", 0.97)]
for x, t, yf in notes:  # 개업 주석은 범례와 겹치지 않게 아래로
    ax.annotate(t, (x, ax.get_ylim()[1] * yf), fontsize=8.5, color=PAL["ink2"],
                ha="left", va="top", rotation=0)
ax.set_ylabel("일 매출 (만원)")
ax.set_title("일별 매출 시계열 (2025-04-03 ~ 2026-04-16) — 회색 음영 = 5일 이상 연속 무매출")
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.margins(x=0.01)
plt.show()

### §2.2 관찰 — 시계열 전도

- 4~5월 안정 → **6~8월 여름 슬럼프**(4~5월의 ¼ 수준) → 9월 개강과 함께 급회복
  → 10월 추석·중간고사 구간 주춤 → 11~12월 회복 → **장기 휴업** → 재개장 후 **2026-03 폭등**(4~5월의 1.8배).
- 재개장 첫날(02-26)은 주문 1건의 소프트 오픈, 이틀 만에 평시 수준으로 정상화, 3월 개강과 함께 본궤도.
- 수준(level)이 구간마다 크게 다름 → 단일 정상성 가정 불가, **구간(regime) 인지가 모델링 전제**.

In [ ]:
# 연속 무매출 구간 표 + 요일별 휴무율(정상 운영기: 개업 후, 장기 휴업 제외)
runs_df = pd.DataFrame([(s.date(), e.date(), n) for s, e, n in runs], columns=["시작", "끝", "일수"])
NOTE = {"2025-04-03": "개업 전 (첫 매출 04-10)", "2025-08-06": "여름 휴가 추정",
        "2025-10-04": "추석 연휴 (10-03 개천절~10-12)", "2025-12-21": "장기 휴업 — 사유 검수 대기"}
runs_df["해석"] = runs_df["시작"].astype(str).map(NOTE).fillna("")
display(runs_df)

normal = full[(full.index >= OPEN_DAY) & ~full.index.isin(pd.date_range(CLOSE_S, CLOSE_E))]
wd = normal.groupby(normal.index.dayofweek).is_open.agg(총일수="size", 영업일="sum")
wd.index = list("월화수목금토일")
wd["휴무일"] = wd["총일수"] - wd["영업일"]
wd["휴무율%"] = (wd["휴무일"] / wd["총일수"] * 100).round(1)
display(wd.T)

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
bars = ax.bar(wd.index, wd["휴무율%"], color=PAL["blue"], width=0.62)
ax.bar_label(bars, fmt="%.0f%%", fontsize=9, color=PAL["ink2"], padding=2)
ax.set_ylabel("휴무율 (%)")
ax.set_title("요일별 휴무율 — 정상 운영기 (개업 후·장기 휴업 제외)")
ax.grid(axis="x", visible=False)
plt.show()

### §2.3 관찰 — 무매출일 구조

- 무매출 123일 = 개업 전 7 + 여름 휴가 6 + 추석 연휴 9 + **장기 휴업 67** + 산발 34일.
- 요일별 휴무율: **일요일 42%**, 토 18%, 평일 7~13% → **고정 정기휴무 요일은 없다.**
  일요일은 "쉬는 날이 많은 요일"일 뿐이라 요일 더미만으로 휴무를 예측할 수 없음 →
  **영업 여부는 예측 대상이 아니라 입력 조건**으로 취급하는 편이 안전 (§9).
- 장기 휴업(2025-12-21~2026-02-25)의 사유·성격(리모델링/업종 개편)은 **담당자 검수 대기** —
  §2.6의 메뉴 교체 증거상 계획된 개편으로 추정되나, 확인 후 M6.A4 학습 구간 설계에 반영.
- 카탈로그(`AI/data/README.md`)의 "휴무 추정 121일 — 요일 패턴 EDA 확인" 항목은 본 절로 해소:
  요일 정기휴무 아님, 위 구조 분해가 확정치.

In [ ]:
# 분포 — 영업일 일 매출 (원 스케일 + log10), 기술통계, IQR 이상치 후보
biz = tot[tot.total_amount > 0].copy()
biz["요일"] = biz.date.dt.dayofweek.map(dict(enumerate("월화수목금토일")))
a = biz.total_amount

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), constrained_layout=True)
axes[0].hist(a / 1e4, bins=34, color=PAL["blue"], edgecolor="white", lw=0.4)
axes[0].axvline(a.median() / 1e4, color=PAL["blue_dark"], lw=1.4, ls="--")
axes[0].annotate(f"중앙값 {a.median()/1e4:,.0f}만", (a.median() / 1e4, axes[0].get_ylim()[1] * 0.92),
                 fontsize=9, color=PAL["ink2"], ha="left")
axes[0].set_xlabel("일 매출 (만원)"); axes[0].set_ylabel("일수"); axes[0].set_title("영업일 일 매출 분포")
axes[1].hist(np.log10(a), bins=34, color=PAL["blue"], edgecolor="white", lw=0.4)
axes[1].set_xlabel("log10(일 매출, 원)"); axes[1].set_title("log10 변환 후")
plt.show()

print(f"n={len(a)} | 평균 {won(a.mean())} | 중앙값 {won(a.median())} | 표준편차 {won(a.std())}")
print(f"최소 {won(a.min())} ~ 최대 {won(a.max())} | 왜도 {a.skew():.2f} | 첨도 {a.kurt():.2f}")

q1, q3 = a.quantile([0.25, 0.75]); iqr = q3 - q1
iqr_tbl = pd.DataFrame(
    [(k, won(q1 - k * iqr), won(q3 + k * iqr), int((a < q1 - k * iqr).sum()), int((a > q3 + k * iqr).sum()))
     for k in (1.5, 3.0)],
    columns=["k", "하한", "상한", "하단 이상치", "상단 이상치"])
display(iqr_tbl)

hi = biz[a > q3 + 1.5 * iqr].sort_values("total_amount", ascending=False)
print(f"IQR k=1.5 상단 {len(hi)}건 (재개장 후 2026-03이 {int((hi.date >= REOPEN).sum())}건):")
display(hi[["date", "요일", "total_amount", "tx_count"]].assign(total_amount=hi.total_amount.map(won)))
lo5 = biz.nsmallest(5, "total_amount")
print("최저 영업일 5건 (전부 주문 1건짜리 날):")
display(lo5[["date", "요일", "total_amount", "tx_count"]].assign(total_amount=lo5.total_amount.map(won)))

biz["객단가"] = biz.total_amount / biz.tx_count
print(f"주문 건수: 중앙값 {biz.tx_count.median():.0f}건/일, 최대 {biz.tx_count.max():.0f}건 "
      f"| 건당 금액: 중앙값 {won(biz['객단가'].median())}, 최대 {won(biz['객단가'].max())}")

### §2.4 관찰 — 분포·이상치

- 우측 왜도 1.49 — log10 변환 후 거의 대칭. **M6.A5 학습 시 log1p 타깃 변환 후보**.
- IQR k=1.5 상단 10건 중 8건이 재개장 후 2026-03에 집중 — "이상치"가 아니라 **새 regime의 정상 수준**.
  나머지 2건(2025-09-11 — 주문 6건뿐인 고액일, 단체 추정, 2025-04-29)만 순수 스파이크.
  하단 이상치 0건 (경계 자체가 음수).
- 최저 영업일들은 전부 주문 1건짜리 날 — 재개장 첫날(02-26) 포함. 오류가 아닌 실제 한산일.
- **k값·처리 방식 확정은 M6.A4에서 train 구간 기준으로** (누수 방지). 현 증거로는
  ① regime 반영(§9) 후 ② log 변환 잔차 기준 k=3.0(또는 winsorize) 조합이 유력 후보.

In [ ]:
# 요일·월 패턴 + 자기상관
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.6), constrained_layout=True)
sns.boxplot(data=biz, x="요일", y=biz.total_amount / 1e4, order=list("월화수목금토일"),
            color=PAL["blue"], width=0.58, fliersize=2.5, ax=axes[0])
axes[0].set_ylabel("일 매출 (만원)"); axes[0].set_xlabel("")
axes[0].set_title("요일별 일 매출 (영업일)")

biz["month"] = biz.date.dt.strftime("%Y-%m")
mm = biz.groupby("month").agg(영업일=("date", "count"), 일평균=("total_amount", "mean"), 월합계=("total_amount", "sum"))
bars = axes[1].bar(mm.index, mm["일평균"] / 1e4, color=PAL["blue"], width=0.62)
axes[1].set_ylabel("일평균 매출 (만원)"); axes[1].set_title("월별 일평균 매출 (영업일 기준)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="x", visible=False)
axes[1].annotate("2026-01 휴업(월 전체) →", (6.6, 115), fontsize=8.5, color=PAL["ink2"], ha="center")
plt.show()

show = mm.copy()
show["일평균"] = show["일평균"].map(won); show["월합계"] = show["월합계"].map(won)
display(show.T)

lag1 = biz.set_index("date").total_amount.autocorr(1)          # 영업일 순차 기준
lag7 = full.total_amount.replace(0, np.nan).autocorr(7)        # 캘린더 7일 전 (무매출일 제외)
print(f"자기상관 — 직전 영업일 {lag1:.2f} | 7일 전(같은 요일) {lag7:.2f}")
print(f"총매출(전 기간): {won(a.sum())} 원")

### §2.5 관찰 — 요일·월 패턴

- **목요일 피크** / **일요일 저점** — 중앙값 2.3배 차. 화·수·목 상승 → 금·토 하락의 대학가 주점 패턴
  (주말엔 학생이 빠지는 상권 특성으로 해석).
- 월별 일평균: 여름(7~8월) 최저 ↔ 3월(재개장+개강) 최고 — 진폭 7배.
- 자기상관: 직전 영업일 0.54, 7일 전 0.47 → **lag1·lag7·rolling7 파생 피처의 근거** (M6.A3).

In [ ]:
# 구조 변화 — 장기 휴업 전/후 비교 (분포·주문 구조·메뉴 구성)
pre_b, post_b = biz[biz.date <= "2025-12-20"], biz[biz.date >= REOPEN]

fig, ax = plt.subplots(figsize=(8.5, 3.4), constrained_layout=True)
bins = np.arange(0, 290, 10)
ax.hist(pre_b.total_amount / 1e4, bins=bins, color=PAL["blue"], alpha=0.62, label=f"휴업 전 (n={len(pre_b)})")
ax.hist(post_b.total_amount / 1e4, bins=bins, color=PAL["orange"], alpha=0.62, label=f"재개장 후 (n={len(post_b)})")
for d, c in [(pre_b, PAL["blue_dark"]), (post_b, PAL["orange"])]:
    ax.axvline(d.total_amount.median() / 1e4, color=c, lw=1.4, ls="--")
ax.set_xlabel("일 매출 (만원)"); ax.set_ylabel("일수")
ax.set_title("휴업 전 vs 재개장 후 일 매출 분포 (점선 = 각 중앙값)")
ax.legend(frameon=False)
plt.show()

cmp = pd.DataFrame({
    "영업일": [len(pre_b), len(post_b)],
    "일 매출 중앙값": [won(pre_b.total_amount.median()), won(post_b.total_amount.median())],
    "일 매출 평균": [won(pre_b.total_amount.mean()), won(post_b.total_amount.mean())],
    "주문 건수 중앙값": [pre_b.tx_count.median(), post_b.tx_count.median()],
    "건당 금액 중앙값": [won(pre_b["객단가"].median()), won(post_b["객단가"].median())],
}, index=["휴업 전", "재개장 후"])
display(cmp)

m_ok = menu[~menu.is_service_item & (menu.amount_net > 0)].dropna(subset=["menu_clean"])
pre_m = set(m_ok[m_ok.date <= "2025-12-20"].menu_clean)
post_m = set(m_ok[m_ok.date >= REOPEN].menu_clean)
new_m, gone_m, kept_m = post_m - pre_m, pre_m - post_m, pre_m & post_m
print(f"판매 메뉴: 휴업 전 {len(pre_m)}종 → 재개장 후 {len(post_m)}종 "
      f"(신규 {len(new_m)} · 단종 {len(gone_m)} · 유지 {len(kept_m)})")
print(f"신규 중 '골목 낙곱새' 시리즈: {sum('골목 낙' in m for m in new_m)}종")
print("신규 예:", sorted(new_m)[:6])
print("단종 예:", sorted(gone_m)[:6])
print("유지 예:", sorted(kept_m)[:6])

### §2.6 관찰 — 장기 휴업 후 구조 변화 (본 EDA 최대 발견)

- 재개장 후 일 매출 중앙값 **2.2배**, 주문 건수 중앙값 7→25건으로 급증.
  건당 금액은 오히려 하락(-39%) — 객수 중심 성장.
- 메뉴 구성이 사실상 교체됨: 휴업 전 54종 → 재개장 후 90종, 그중 **신규 69종**·유지 21종. 간판 신메뉴는
  '골목 낙곱새/낙우새/낙곱닭' 시리즈(크기 변형 포함 20종). 휴업 전 주력이던 마라 계열(마라샹궈 등)·중식 안주
  다수 단종 → **마라 주점 → 낙곱새 전문점 업종 개편**.
- 함의: 휴업 전/후는 **다른 데이터 생성 과정(regime)**. 재개장 후 표본은 49일뿐이라 post-only 학습은 불가.
  → §9의 학습 구간 3안 비교로 이관. 휴업 사유·운영 변경의 사실 확인(검수 항목)이 선행 조건.

## §3 메뉴 차원 — 집중도·표기 품질·정합성

카탈로그의 "메뉴명 통합 매핑 — 확정은 EDA에서" 항목을 본 절에서 처리한다.

In [ ]:
# 파레토(매출 상위 20) + 누적 커버리지 — 이중축 대신 상하 분리
top = m_ok.groupby("menu_clean").amount_net.sum().sort_values(ascending=False)
cum = top.cumsum() / top.sum()
n80 = int((cum <= 0.80).sum()) + 1
t20 = top.head(20)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.5, 5.6), sharex=True, constrained_layout=True,
                               height_ratios=[2, 1])
ax1.bar(range(len(t20)), t20.values / 1e4, color=PAL["blue"], width=0.62)
ax1.set_ylabel("총매출 (만원)")
ax1.set_title(f"메뉴별 매출 상위 20 — 전체 {len(top)}종 중 상위 {n80}종이 매출 80% 커버")
ax1.grid(axis="x", visible=False)
ax2.plot(range(len(t20)), cum.head(20).values * 100, color=PAL["orange"], lw=2, marker="o", ms=4)
ax2.axhline(80, color=PAL["ink2"], lw=1, ls="--")
ax2.annotate("80%", (0.2, 81.5), fontsize=9, color=PAL["ink2"])
ax2.set_ylabel("누적 비중 (%)")
ax2.set_xticks(range(len(t20)))
ax2.set_xticklabels([m[:14] for m in t20.index], rotation=45, ha="right", fontsize=8.5)
plt.show()

cat = m_ok.groupby("category").amount_net.sum().sort_values(ascending=False)
cat_pct = (cat / cat.sum() * 100).round(1)
fig, ax = plt.subplots(figsize=(7.5, 3.2), constrained_layout=True)
ax.barh(cat_pct.index[::-1], cat_pct.values[::-1], color=PAL["blue"], height=0.6)
ax.set_xlabel("매출 비중 (%)"); ax.set_title("카테고리별 매출 비중 — 명칭 비일관 주의(안주/안주류 등)")
ax.grid(axis="y", visible=False)
for y, v in enumerate(cat_pct.values[::-1]):
    ax.annotate(f"{v}", (v + 0.4, y), va="center", fontsize=8.5, color=PAL["ink2"])
plt.show()

In [ ]:
# 표기 품질·정합성 점검
import re

svc = menu[menu.is_service_item]
print(f"서비스성 행 {len(svc)}건 (배달료 등) — 합계 {won(svc.amount_net.sum())} 원")

nan_mc = menu[menu.menu_clean.isna()]
print(f"\nmenu_clean NaN {len(nan_mc)}행 — 전부 재개장 후 대괄호 사이드 품목:")
display(nan_mc[["date", "menu_raw", "category", "qty", "amount_net"]])

norm = lambda s: re.sub(r"[\s'\"‘’“”\[\]()]+", "", str(s)).lower()
g = m_ok.groupby("menu_clean").amount_net.sum().reset_index()
g["정규화키"] = g.menu_clean.map(norm)
dup = g[g.duplicated("정규화키", keep=False)].sort_values(["정규화키", "amount_net"], ascending=[True, False])
print("표기 이형(공백·따옴표 제거 시 충돌) 그룹:")
display(dup.assign(amount_net=dup.amount_net.map(won)))

# 일계 vs 메뉴 합 정합성 — 서비스행 포함 기준이 실제 리포트 구조와 일치
for label, sub in [("서비스행 제외", menu[~menu.is_service_item]), ("전체 행", menu)]:
    ms = sub.groupby("date").amount_net.sum()
    d = tot.set_index("date").join(ms.rename("s"), how="left")
    diff = d.total_amount - d.s.fillna(0)
    print(f"{label:8s}: 1원 초과 차이 {int((diff.abs() > 1).sum())}일 "
          f"(최대 +{diff.max():,.0f} / {diff.min():,.0f})")

### §3 관찰 + 메뉴 매핑 확정안

- **집중도** — 매출>0 메뉴 123종 중 **상위 14종이 매출 80%**. 1위 소주(메뉴 매출의 24%),
  2위 마라 전골, 3위 맥주. 카테고리로는 술 39.6% + 국물 28.1% + 안주 24.8% — 주류 동반 판매 업태 확정.
- **정합성** — 일계 vs 메뉴합은 *서비스행(배달료) 포함* 기준 수록 258일 중 11일만 1원 초과 차이(소액 — 전표 단위 할인 추정) → 메뉴 테이블을 일계의 분해로 신뢰 가능.
- **매핑 확정안** (M6.A3 데이터 준비에서 `sales_transform.py`에 반영):
  1. `마라전골` → `마라 전골` 병합 — 유일한 표기 이형 충돌(소액).
  2. `[쌈무]`·`[파김치]`·`[공깃밥]` 등 대괄호 품목(7행, 재개장 후 POS 포맷) — 대괄호 제거 규칙 추가,
     category `사이드` 부여 (유상 판매라 서비스 플래그는 아님).
  3. 카테고리 표준화 맵(`안주류→안주`, `식사류→식사`, `시원한거→음료` 등) — 재개장 후 POS 개편으로
     생긴 비일관, 메뉴 레벨 피처를 쓰게 되면 M6.A3에서 적용.
- 메뉴-일 레벨 모델링(메뉴별 수요 예측)은 재개장 후 표본 부족(신규 메뉴 46일 이하)이므로
  **MVP 타깃은 일 매출 총액 유지**, 메뉴 레벨은 카테고리 집계 피처로만 활용 권고.

## §4 기상 — 세종연서(611) 관측소

In [ ]:
w611 = wx[wx.station_id == 611].set_index("date").sort_index()
win = w611.loc[CAL.min():CAL.max()]
cols = ["temp_avg", "temp_min", "temp_max", "precip_mm", "wind_avg", "wind_max_inst"]

na_tbl = pd.DataFrame({
    "전체 기간 NA (2020~)": w611[cols].isna().sum(),
    "판매 기간 NA": win[cols].isna().sum(),
})
na_tbl.loc["결측 일자(행 없음)"] = [len(pd.date_range(w611.index.min(), w611.index.max()).difference(w611.index)),
                                len(CAL.difference(win.index))]
display(na_tbl)
print("판매 기간 내 NA 발생일:", win[win[cols].isna().any(axis=1)].index.strftime("%Y-%m-%d").tolist())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12.5, 4.6), sharex=True, constrained_layout=True,
                               height_ratios=[2, 1])
for s, e, n in runs:
    for ax in (ax1, ax2):
        ax.axvspan(s, e + pd.Timedelta(days=1), color=PAL["gray"], alpha=0.45, zorder=0)
ax1.plot(win.index, win.temp_avg, lw=1.1, color=PAL["blue"], label="평균기온")
ax1.fill_between(win.index, win.temp_min, win.temp_max, color=PAL["blue"], alpha=0.18, lw=0)
ax1.set_ylabel("기온 (°C)"); ax1.legend(frameon=False, fontsize=9, loc="upper left")
ax1.set_title("판매 기간 기상 — 기온(위, 음영=일교차 범위)·강수(아래), 회색=매장 무매출 구간")
ax2.bar(win.index, win.precip_mm, color=PAL["blue"], width=1)
ax2.set_ylabel("강수 (mm)")
plt.show()
print(f"판매 기간: 평균기온 {win.temp_avg.min():.1f} ~ {win.temp_avg.max():.1f}°C, "
      f"강수>0 {int((win.precip_mm > 0).sum())}일 ({(win.precip_mm > 0).mean()*100:.0f}%)")

### §4 관찰 — 기상

- 판매 기간 379일 **전일 커버**, NA는 강수 1일·바람 2일뿐(기온 0) → **선형 보간으로 충분** (규칙 확정은 M6.A4).
- 평균기온 -8.8~29.4°C 로 사계절 스윙 큼, 비 온 날 24% — 파생 피처(더위/추위 구간, 비 여부)는 M6.A3에서.

## §5 공휴일·학사일정 — 일별 플래그 구성과 매출 차이

In [ ]:
# 판매 기간 내 공휴일 — 영업 여부·매출
hw = hol[hol.date.isin(CAL)].copy()
hw = hw.merge(tot[["date", "total_amount"]], on="date", how="left")
hw["영업"] = hw.total_amount.fillna(0) > 0
hw["매출"] = hw.total_amount.fillna(0).map(won)
hw["정상운영기"] = hw.date.isin(normal.index)  # False = 개업 전·장기 휴업 중
display(hw[["date", "holiday_name", "영업", "매출", "정상운영기"]])
hn, nh = hw[hw["정상운영기"]], normal[~normal.index.isin(hol.date)]
print(f"공휴일 영업률(정상 운영기 {len(hn)}일 기준) {hn['영업'].mean()*100:.0f}% "
      f"vs 비공휴일 {nh.is_open.mean()*100:.0f}% "
      f"| 영업한 공휴일 매출 중앙값 {won(hw.loc[hw['영업'], 'total_amount'].median())}")

# 학사일정 → 일별 플래그 (판매 기간)
flags = pd.DataFrame(index=CAL, data={"is_semester": False, "is_exam": False, "is_session": False})
for _, r in acad.iterrows():
    sl = flags.loc[max(r.start_date, CAL[0]):min(r.end_date, CAL[-1])]
    if len(sl) == 0:
        continue
    if r.event == "semester":
        flags.loc[sl.index, "is_semester"] = True
    elif "exam" in r.event:
        flags.loc[sl.index, "is_exam"] = True
    elif r.event in ("summer_session", "winter_session"):
        flags.loc[sl.index, "is_session"] = True

seg = np.where(flags.is_exam, "시험주간", np.where(flags.is_semester, "학기중(비시험)", "방학·기타"))
ob_seg = full.join(pd.Series(seg, index=CAL, name="구간"))
ob_seg = ob_seg[ob_seg.is_open]
order = ["방학·기타", "학기중(비시험)", "시험주간"]
fig, ax = plt.subplots(figsize=(7.5, 3.4), constrained_layout=True)
sns.boxplot(data=ob_seg, x="구간", y=ob_seg.total_amount / 1e4, order=order,
            color=PAL["blue"], width=0.5, fliersize=2.5, ax=ax)
ax.set_ylabel("일 매출 (만원)"); ax.set_xlabel("")
ax.set_title("학사 구간별 일 매출 (영업일, 전 기간)")
plt.show()
stat = ob_seg.groupby("구간").total_amount.agg(["count", "median", "mean"]).loc[order]
display(stat.assign(median=stat["median"].map(won), mean=stat["mean"].map(won)))

### §5 관찰 — 공휴일·학사

- **공휴일엔 잘 쉰다** — 정상 운영기 공휴일 14일 중 7일만 영업(50%, 비공휴일 86%), 영업해도 평시의 절반 이하로 저조
  → `is_holiday`는 영업 여부·매출 양쪽에 걸린 피처.
- **학기 효과가 외부 변수 중 최대** — 학기중(비시험) 중앙값이 방학·기타의 2.6배 (전 기간 — 3월 재개장 효과 포함 주의).
  휴업 전만 보면 **2.2배**.
- **시험주간 억제는 뚜렷** — 시험주간 22일(전부 휴업 전 구간)의 중앙값은 휴업 전 학기중(비시험)의 **66% 수준**. 단 시험 일자 자체가 estimated(8주차/15주차 규칙)라 **검수(재학생 확인) 후 확정**.
- 겨울 계절학기(2025-12-23~2026-01-15)는 장기 휴업과 완전히 겹쳐 이번 데이터로는 효과 식별 불가.

## §6 유동인구 (월간 대체 데이터) — 조치원읍

In [ ]:
jc = pop[pop.dong == "조치원읍"].copy()
jc["month_ts"] = pd.to_datetime(jc.month + "-01")
all_m = pd.date_range("2025-01-01", "2026-05-01", freq="MS")
jc = jc.set_index("month_ts").reindex(all_m)

# 유동(30만대)과 생활(4만대)은 스케일이 7배 차이 — 같은 축 대신 상하 분리
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10.5, 4.8), sharex=True, constrained_layout=True,
                               height_ratios=[1.4, 1])
ax1.plot(jc.index, jc.floating_pop / 1e4, color=PAL["blue"], lw=2, marker="o", ms=5, label="유동인구")
miss = jc[jc.floating_pop.isna()].index
ax1.scatter(miss, [21.5] * len(miss), marker="x", s=42, color=PAL["ink2"], label="누락 월 (7개)")
ax1.set_ylabel("유동인구 (만명)")
ax1.set_title("조치원읍 월간 인구 — 결측 월은 선 끊김·× 표시")
ax1.legend(frameon=False, fontsize=9, loc="lower right")
ax2.plot(jc.index, jc.living_pop.replace(0, np.nan) / 1e4, color=PAL["orange"], lw=2, marker="o", ms=5)
ax2.annotate("2025-08 생활인구=0 (원본 오류 — 검수 대기)", (pd.Timestamp("2025-08-01"), 4.05),
             fontsize=8.5, color=PAL["ink2"], ha="center")
ax2.set_ylabel("생활인구 (만명)")
ax2.set_ylim(3.4, 5.0)
plt.show()

# 월 일평균 매출과의 관계 (겹치는 월만 — 참고용)
msales = biz.groupby("month").total_amount.mean().rename("일평균매출")
jm = pop[pop.dong == "조치원읍"].set_index("month").join(msales, how="inner")
jm = jm.dropna(subset=["floating_pop", "일평균매출"])
rho = jm.floating_pop.corr(jm.일평균매출, method="spearman")
fig, ax = plt.subplots(figsize=(5.6, 3.6), constrained_layout=True)
ax.scatter(jm.floating_pop / 1e4, jm.일평균매출 / 1e4, s=42, color=PAL["blue"])
for m, r in jm.iterrows():
    ax.annotate(m, (r.floating_pop / 1e4, r.일평균매출 / 1e4), fontsize=8, color=PAL["ink2"],
                xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("유동인구 (만명/월)"); ax.set_ylabel("일평균 매출 (만원)")
ax.set_title(f"유동인구 vs 월 일평균 매출 — n={len(jm)}, Spearman {rho:.2f}")
plt.show()

### §6 관찰 — 유동인구

- 조치원읍 유동인구에 학기 리듬이 그대로 보임 — 3월(개강) 35만 피크 ↔ 1월(방학) 23만 저점.
- 판매 기간과 겹치면서 값이 있는 월은 **7개뿐**(n=7, Spearman 0.64) — 방향성 참고용이지 통계적 근거는 아님.
- 활용 방침(M6.A3): 월간 → 일별 조인 시 **전월(lag 1M) 값 사용**(당월 값은 월말까지 확정되지 않아 누수),
  누락 7개월은 M6.A4에서 보간(선형 후보). **학기 플래그와 정보 중복 가능성 높아 1차 피처 선별 대상.**
- 2025-08 생활인구 0 — 원본 시트 오류로 판단(같은 달 유동인구는 정상). 검수 대기, 보간 처리 예정.

## §7 타깃-피처 관계 요약 — 전 기간 vs 휴업 전

**해석 원칙** — 재개장 효과(2026-03 폭등)가 3월의 기온·개강과 동시에 움직여 전 기간 상관에는
regime 교란이 섞인다. 따라서 **휴업 전(2025-04-10~12-20) 구간의 상관을 기준**으로 삼고 전 기간은 참고만.

In [ ]:
mdf = full.join(w611[["temp_avg", "temp_max", "precip_mm"]]).join(flags)
mdf["is_rain"] = mdf.precip_mm > 0
mdf["is_holiday"] = mdf.index.isin(hol.date)
ob = mdf[mdf.is_open]
ob_pre = ob.loc[OPEN_DAY:"2025-12-20"]

VARS = ["temp_avg", "temp_max", "precip_mm", "is_rain", "is_holiday", "is_semester", "is_exam"]
corr_tbl = pd.DataFrame({
    ("Spearman", "휴업 전"): [ob_pre.total_amount.corr(ob_pre[c].astype(float), method="spearman") for c in VARS],
    ("Spearman", "전 기간"): [ob.total_amount.corr(ob[c].astype(float), method="spearman") for c in VARS],
    ("Pearson", "휴업 전"): [ob_pre.total_amount.corr(ob_pre[c].astype(float)) for c in VARS],
    ("Pearson", "전 기간"): [ob.total_amount.corr(ob[c].astype(float)) for c in VARS],
}, index=VARS).round(3)
display(corr_tbl)

fig, ax = plt.subplots(figsize=(7.2, 4), constrained_layout=True)
for flag, label, color in [(True, "학기중", PAL["blue"]), (False, "그 외(방학·계절학기)", PAL["orange"])]:
    d = ob[ob.is_semester == flag]
    ax.scatter(d.temp_avg, d.total_amount / 1e4, s=17, alpha=0.7, color=color, label=label)
ax.set_xlabel("평균기온 (°C)"); ax.set_ylabel("일 매출 (만원)")
ax.set_title("기온 vs 일 매출 — 학기 여부로 분리하면 기온 단독 효과는 약해짐")
ax.legend(frameon=False, fontsize=9)
plt.show()

### §7 관찰 — 관계 요약

| 변수 | 휴업 전 Spearman | 판정 |
|---|---|---|
| is_semester | **+0.36** | 외부 변수 중 최강. 파생(개강 주, 학기 진행률) 가치 있음 |
| temp_avg | -0.35 | 여름 방학과 동행하는 **교란된 상관** — 산점도에서 학기 분리 시 기울기 완만. 다변량에서 재평가 |
| is_holiday | -0.19 | 영업 여부와 매출 양쪽에 작용 |
| is_exam | -0.07 | 단변량 상관은 약해 보이나 방학 저점과 섞인 착시 — 구간 비교(§5)로는 학기중 대비 34%↓. 검수 후 재평가 |
| is_rain / precip_mm | +0.06 / +0.05 | **억제 효과 없음** — 비 오는 날 매출이 오히려 소폭 높음. 우선순위 낮음 |

- 요일은 §2.5에서 확인(목 피크·일 저점) — 순서형이 아니라 더미로 다룬다.
- 단변량 상관은 참고 지표일 뿐 — 피처 선별은 M6.A3에서 다변량(트리 중요도·permutation) 기준으로.

## §8 결측·이상치 정량 총괄표

In [ ]:
summary = pd.DataFrame([
    {"소스": "판매 일계 (타깃)", "결측·공백": f"무매출 {int((~full.is_open).sum())}일 / 캘린더 {len(CAL)}일 — 구조 분해 §2.3",
     "이상 신호": f"IQR k=1.5 상단 {len(hi)}건(2026-03에 {int((hi.date >= REOPEN).sum())}건), 하단 0건; POS 테스트일 1건(2025-04-03)",
     "방침(M6.A4 확정)": "영업일만 학습, regime 처리 §9, k·변환은 train 기준 확정"},
    {"소스": "판매 메뉴", "결측·공백": f"menu_clean NaN {int(menu.menu_clean.isna().sum())}행, 표기 이형 1그룹",
     "이상 신호": "카테고리 명칭 비일관(안주/안주류 등)", "방침(M6.A4 확정)": "매핑 확정안 §3 → sales_transform.py 반영(M6.A3)"},
    {"소스": "기상 611", "결측·공백": f"판매 기간 NA: 강수 {int(win.precip_mm.isna().sum())}·바람 {int(win[['wind_avg','wind_max_inst']].isna().any(axis=1).sum())}일, 기온 0",
     "이상 신호": "없음", "방침(M6.A4 확정)": "선형 보간 후보 — train 구간 기준 확정"},
    {"소스": "공휴일", "결측·공백": "없음 (수동 추가 1건 검수 대기: 2025-10-10)",
     "이상 신호": "—", "방침(M6.A4 확정)": "그대로 사용"},
    {"소스": "학사일정", "결측·공백": "시험주간 일자 estimated (검수 대기)",
     "이상 신호": "겨울 계절학기 구간이 휴업과 완전 중첩", "방침(M6.A4 확정)": "검수 후 is_exam 재산정"},
    {"소스": "유동인구 (월간)", "결측·공백": "유동인구 누락 7개월, 2025-08 생활인구 0(원본 오류)",
     "이상 신호": "—", "방침(M6.A4 확정)": "전월 lag 조인 + 누락 월 보간, 1차 피처 선별 대상"},
])
summary.set_index("소스")

## §9 판정·다음 단계

**M6.A2 종료 판정** — 모든 입력 변수의 분포·결측·이상치 정량 완료(위 §8), 타깃-피처 관계 확인 완료. **EDA 보고서 1회분 산출 충족.**

### 학습 구간 설계 — 3안 (M6.A4~A5에서 확정, 담당자 휴업 사유 검수가 선행)

| 안 | 내용 | 평가 |
|---|---|---|
| ① 전 기간 + regime 피처 | 전체 256 영업일 학습, `is_post_renewal`(+재개장 후 경과일) 피처 추가 | **권고안** — 표본 최대 활용, 트리 모델이 분기 학습 |
| ② 재개장 후만 | 2026-02-26 이후 49일만 학습 | 표본 부족으로 기각(요일×학기 조합 커버 불가) |
| ③ 전 기간 학습 + 최근 가중 | 시간 가중(감쇠) 적용 | ①의 보조 실험으로만 |

- Walk-forward CV(TimeSeriesSplit)에서 **휴업 구간(2025-12-21~2026-02-25)이 통째로 검증 fold가 되지 않도록** fold 경계 설계 필요.
- 예측 서비스 관점: 예측 대상은 **영업일의 매출**. 영업 여부(휴무)는 예측하지 않고 입력 조건으로 받는다(§2.3 근거 — 휴무가 규칙적이지 않음).

### M6.A3 피처 후보 시사점 (본 EDA 근거)

- **시간**: 요일 더미(목·금·토·일 구분 필수), lag1(직전 영업일)·lag7·rolling7/14 (§2.5 자기상관 0.54/0.47)
- **학사**: is_semester(+개강 첫 주 플래그 — 3월·9월 스파이크), is_exam(검수 후), 학기 진행률
- **달력**: is_holiday·연휴 여부, 월(계절성)
- **기상**: temp 구간화(더위/추위) — 단독 선형 효과는 교란돼 있으므로(§7) 다변량에서 판정. 강수는 후순위
- **regime**: is_post_renewal + 재개장 후 경과일 (①안)
- **유동인구**: 전월 lag 값 (학기 플래그와 중복성 검사 후 채택 여부 결정)

### 검수 대기 항목 (담당자 — `AI/data/README.md` 검수 섹션과 동일)

1. **장기 휴업(2025-12-21~2026-02-25) 사유** — 본 EDA는 메뉴 교체 증거로 "계획된 업종 개편"으로 추정(§2.6). 사실 확인 필요
2. 유동인구 누락 7개월 원본 제공 여부 + 2025-08 생활인구 0 오류
3. 학사일정 시험주간(estimated) 실제 일자
4. holidays 2025-10-10 임시공휴일 지정 여부

**다음 마일스톤** — M6.A3 피처 엔지니어링 (`02_features.ipynb`): 위 후보 30~50개 생성 → 상관·VIF·트리 중요도 1차 선별.